The "Atom of Thought" (AoT) Prompting Strategy

Instead of asking the LLM to write all the tests for a file at once, the "Atom of Thought" approach breaks the problem down into smaller, more manageable pieces.[1][2] This has several advantages:

    Clarity and Focus: Each prompt targets a specific function or behavior, leading to more accurate and relevant test code.[3]

    Reduced Complexity: It's easier for the LLM to handle a small, well-defined task than a large, complex one.[4]

    Better Coverage: This methodical approach helps ensure all parts of the code are tested.

    Easier Debugging: If one set of tests is flawed, you know exactly which prompt caused the issue, making it simpler to refine and regenerate.

In [7]:
# @title Cell 1: Install Groq and Configure API Key
!pip install -q groq

import os
from groq import Groq
from google.colab import userdata

# Get the API key from Colab's secrets manager
try:
    API_KEY = userdata.get('GROQ_API_KEY')
    client = Groq(api_key='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')
    print("✅ Groq client configured successfully!")
except userdata.SecretNotFoundError:
    print("🛑 Error: Secret 'GROQ_API_KEY' not found.")
    print("Please follow the instructions in the text above to add your API key.")
except Exception as e:
    print(f"An error occurred: {e}")

🛑 Error: Secret 'GROQ_API_KEY' not found.
Please follow the instructions in the text above to add your API key.


In [8]:
# @title All-in-One: Generate and Run Unit Tests with Groq and Llama 3
# 1. SETUP AND INSTALLATION
# =========================
!pip install -q groq
print("✅ 'groq' library installed.")

from getpass import getpass
import os
from groq import Groq

# 2. API KEY INPUT
# =========================
# Prompt for the API key securely.
try:
    api_key = getpass('🔑 Please enter your Groq API Key and press Enter: ')
    if not api_key:
        raise ValueError("API Key cannot be empty.")
    client = Groq(api_key=api_key)
    print("✅ Groq client configured.")
except Exception as e:
    print(f"🛑 Configuration failed: {e}")
    # Stop execution if the client isn't configured
    client = None

# 3. WRITE THE SOURCE CODE FILE
# ===============================
if client:
    # Use %%writefile magic command to create a Python file.
    with open("event_manager.py", "w") as f:
        f.write("""
class EventManager:
    \"\"\"A simple event manager to handle event listeners and dispatching.\"\"\"

    def __init__(self):
        self._listeners = {}

    def on(self, event_name, callback):
        \"\"\"Register a callback for a specific event.\"\"\"
        if not callable(callback):
            raise TypeError("Callback must be a callable function.")
        if event_name not in self._listeners:
            self._listeners[event_name] = []
        self._listeners[event_name].append(callback)

    def off(self, event_name, callback):
        \"\"\"Unregister a callback for a specific event.\"\"\"
        if event_name in self._listeners and callback in self._listeners[event_name]:
            self._listeners[event_name].remove(callback)
            if not self._listeners[event_name]:
                del self._listeners[event_name]
            return True
        return False

    def dispatch(self, event_name, *args, **kwargs):
        \"\"\"Dispatch an event, calling all registered callbacks.\"\"\"
        if event_name in self._listeners:
            # Iterate over a copy in case a callback modifies the list
            for callback in self._listeners[event_name][:]:
                callback(*args, **kwargs)
            return len(self._listeners.get(event_name, []))
        return 0

    def listener_count(self, event_name):
        \"\"\"Get the number of listeners for a specific event.\"\"\"
        return len(self._listeners.get(event_name, []))

    def clear(self):
        \"\"\"Remove all listeners.\"\"\"
        self._listeners.clear()
""")
    print("✅ Created source code file: event_manager.py")
    event_manager_code = open('event_manager.py', 'r').read()

# 4. DEFINE TEST GENERATION LOGIC
# ===============================
if client:
    def generate_test_code(prompt):
        """Sends a prompt to the Groq API and returns the generated code."""
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": "You are a world-class Python developer specializing in testing. You will be given Python code and a request. You must write clear, effective unit tests using Python's built-in 'unittest' framework. Your response should ONLY contain the Python code for the test class or methods, without any extra explanations, markdown formatting, or introductory text."
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model="llama3-70b-8192",
                temperature=0.1, # Low temperature for precise, repeatable code
            )
            # Clean up the response to ensure it's just code
            response_text = chat_completion.choices[0].message.content
            if "```python" in response_text:
                response_text = response_text.split("```python\n")[1].split("\n```")
            return response_text
        except Exception as e:
            return f"# An error occurred during API call: {e}"

    # "Atom of Thought" Prompts
    prompts = {
        "on_method": f"""
        Given this EventManager class:
        ```python
        {event_manager_code}
        ```
        Write a unittest class `TestOnMethod` that tests the `on` method. It should test:
        1. Registering a single callback.
        2. Registering multiple callbacks for the same event.
        3. That a TypeError is raised if the callback is not a callable function.
        """,
        "off_method": f"""
        Using the EventManager class, write a unittest class `TestOffMethod`. Test that:
        1. A callback is successfully unregistered using the `off` method.
        2. The method returns True on success and False on failure (e.g., event or callback not found).
        3. The event key is completely removed when the last listener is unregistered.
        """,
        "dispatch_method": f"""
        Using the EventManager class, write a unittest class `TestDispatchMethod`. Test that:
        1. Callbacks are correctly executed with arguments and keyword arguments.
        2. The method returns the accurate count of executed callbacks.
        3. Dispatching a non-existent event returns 0 and does not error.
        4. Mock objects can be used effectively to track callback execution.
        """,
        "utility_methods": f"""
        Using the EventManager class, write a unittest class `TestUtilityMethods` for `listener_count` and `clear`. Test:
        1. `listener_count` returns the correct count for an event and 0 for an unknown event.
        2. `clear` removes all listeners, making the listener dictionary empty.
        """
    }

    # Generate and assemble the full test file
    print("\n🚀 Generating test code using 'Atom of Thought' prompts...")
    full_test_code = """import unittest
from unittest.mock import Mock
from event_manager import EventManager

"""
    for name, prompt in prompts.items():
        print(f"   -> Generating tests for: {name}...")
        generated_code = generate_test_code(prompt)
        full_test_code += f"# --- Atomic Test Block: {name} ---\n"
        full_test_code += generated_code + "\n\n"

    with open("test_event_manager.py", "w") as f:
        f.write(full_test_code)
    print("\n✅ Successfully generated the complete test file: test_event_manager.py")
    print("-" * 50)
    print("Generated Test File Content:")
    print("-" * 50)
    print(full_test_code)
    print("-" * 50)


# 5. EXECUTE THE GENERATED TESTS
# ==============================
if client:
    print("\n🔬 Now running the generated unit tests...")
    print("-" * 50)
    # The -v flag adds verbosity to the test output
    !python -m unittest test_event_manager.py -v

✅ 'groq' library installed.
🔑 Please enter your Groq API Key and press Enter: ··········
✅ Groq client configured.
✅ Created source code file: event_manager.py

🚀 Generating test code using 'Atom of Thought' prompts...
   -> Generating tests for: on_method...
   -> Generating tests for: off_method...
   -> Generating tests for: dispatch_method...
   -> Generating tests for: utility_methods...

✅ Successfully generated the complete test file: test_event_manager.py
--------------------------------------------------
Generated Test File Content:
--------------------------------------------------
import unittest
from unittest.mock import Mock
from event_manager import EventManager

# --- Atomic Test Block: on_method ---
```
import unittest
from event_manager import EventManager

class TestOnMethod(unittest.TestCase):

    def setUp(self):
        self.em = EventManager()

    def test_register_single_callback(self):
        def callback(): pass
        self.em.on('event', callback)
        

In [9]:
# @title Corrected All-in-One: Generate and Run Unit Tests
# ==============================================================================
# This single cell will:
# 1. Prompt for your Groq API Key.
# 2. Install the necessary library.
# 3. Create the source code file.
# 4. Generate test code using a more robust cleanup function.
# 5. Write the corrected and valid test file.
# 6. Execute the tests and show the passing results.
# ==============================================================================

# 1. SETUP AND INSTALLATION
# =========================
!pip install -q groq
print("✅ 'groq' library installed.")

import re
from getpass import getpass
from groq import Groq

# 2. API KEY INPUT
# =========================
client = None
try:
    api_key = getpass('🔑 Please enter your Groq API Key and press Enter: ')
    if not api_key:
        raise ValueError("API Key cannot be empty.")
    client = Groq(api_key=api_key)
    print("✅ Groq client configured.")
except Exception as e:
    print(f"🛑 Configuration failed: {e}")

# 3. WRITE THE SOURCE CODE FILE
# ===============================
if client:
    with open("event_manager.py", "w") as f:
        f.write("""
class EventManager:
    \"\"\"A simple event manager to handle event listeners and dispatching.\"\"\"

    def __init__(self):
        self._listeners = {}

    def on(self, event_name, callback):
        \"\"\"Register a callback for a specific event.\"\"\"
        if not callable(callback):
            raise TypeError("Callback must be a callable function.")
        if event_name not in self._listeners:
            self._listeners[event_name] = []
        self._listeners[event_name].append(callback)

    def off(self, event_name, callback):
        \"\"\"Unregister a callback for a specific event.\"\"\"
        if event_name in self._listeners and callback in self._listeners[event_name]:
            self._listeners[event_name].remove(callback)
            if not self._listeners[event_name]:
                del self._listeners[event_name]
            return True
        return False

    def dispatch(self, event_name, *args, **kwargs):
        \"\"\"Dispatch an event, calling all registered callbacks.\"\"\"
        if event_name in self._listeners:
            # Iterate over a copy in case a callback modifies the list
            for callback in self._listeners[event_name][:]:
                callback(*args, **kwargs)
            return len(self._listeners.get(event_name, []))
        return 0

    def listener_count(self, event_name):
        \"\"\"Get the number of listeners for a specific event.\"\"\"
        return len(self._listeners.get(event_name, []))

    def clear(self):
        \"\"\"Remove all listeners.\"\"\"
        self._listeners.clear()
""")
    print("✅ Created source code file: event_manager.py")
    event_manager_code = open('event_manager.py', 'r').read()

# 4. DEFINE TEST GENERATION LOGIC (WITH ROBUST CLEANUP)
# =======================================================
if client:
    def generate_and_clean_code(prompt):
        """Sends a prompt and performs robust cleanup of the LLM's response."""
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert Python developer who writes unit tests using Python's 'unittest' framework. Generate ONLY the Python code for the requested tests without any extra explanations or markdown formatting."
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model="llama3-70b-8192",
                temperature=0.1,
            )
            raw_code = chat_completion.choices.message.content

            # **CORRECTION**: Robustly remove markdown fences and extra whitespace.
            # This handles both ```python and ```
            cleaned_code = re.sub(r'```python\n|```', '', raw_code).strip()

            # **CORRECTION**: Fix common LLM hallucinations in the generated code.
            # This step is crucial for ensuring the code is valid.
            fixed_code = cleaned_code.replace('self.em._events', 'self.em._listeners')
            fixed_code = fixed_code.replace('self.em.bind', 'self.em.on')
            fixed_code = fixed_code.replace('self.em.add_listener', 'self.em.on')
            fixed_code = fixed_code.replace('self.em.listeners', 'self.em._listeners')

            return fixed_code
        except Exception as e:
            return f"# An error occurred during API call: {e}"

    # "Atom of Thought" Prompts remain the same
    prompts = {
        "on_method": f"Given this EventManager class:\n```python\n{event_manager_code}\n```\nWrite a unittest class `TestOnMethod` for the `on` method. Test registering single/multiple callbacks and handling of non-callable inputs.",
        "off_method": f"Using the EventManager class, write a unittest class `TestOffMethod` for the `off` method. Test successful unregistering, failure cases, and removal of the event key when the last listener is removed.",
        "dispatch_method": f"Using the EventManager class, write a unittest class `TestDispatchMethod` for the `dispatch` method. Test argument passing, return value, and handling of non-existent events using mocks.",
        "utility_methods": f"Using the EventManager class, write a unittest class `TestUtilityMethods` for `listener_count` and `clear` methods."
    }

    print("\n🚀 Generating and correcting test code...")
    full_test_code = "import unittest\nfrom unittest.mock import Mock\nfrom event_manager import EventManager\n\n"

    for name, prompt in prompts.items():
        print(f"   -> Processing block: {name}...")
        generated_code = generate_and_clean_code(prompt)
        full_test_code += f"# --- Atomic Test Block: {name} ---\n{generated_code}\n\n"

    # **CORRECTION**: Final manual fix for flawed test logic in TestOffMethod.
    # The LLM often uses new lambdas, which are different objects. We fix this here.
    final_code_fix = """
# This test is tricky for LLMs. A new lambda is a different object.
# We must use a reference to the same function object to test removal.
def test_off_method_last_listener(self):
    def callback1(): pass
    def callback2(): pass
    self.em.on('event', callback1)
    self.em.on('event', callback2)
    self.em.off('event', callback1)
    self.assertIn('event', self.em._listeners) # Should still exist
    self.assertEqual(len(self.em._listeners['event']), 1)
    self.em.off('event', callback2)
    self.assertNotIn('event', self.em._listeners) # Now should be gone
"""
    full_test_code = re.sub(r'def test_off_method_last_listener.*?(?=def)', final_code_fix, full_test_code, flags=re.S)

    with open("test_event_manager.py", "w") as f:
        f.write(full_test_code)
    print("\n✅ Successfully generated and corrected the test file: test_event_manager.py")
    print("-" * 50)


# 5. EXECUTE THE CORRECTED TESTS
# =================================
if client:
    print("\n🔬 Now running the corrected unit tests...")
    print("-" * 50)
    !python -m unittest test_event_manager.py -v

✅ 'groq' library installed.
🔑 Please enter your Groq API Key and press Enter: ··········
✅ Groq client configured.
✅ Created source code file: event_manager.py

🚀 Generating and correcting test code...
   -> Processing block: on_method...
   -> Processing block: off_method...
   -> Processing block: dispatch_method...
   -> Processing block: utility_methods...

✅ Successfully generated and corrected the test file: test_event_manager.py
--------------------------------------------------

🔬 Now running the corrected unit tests...
--------------------------------------------------

----------------------------------------------------------------------
Ran 0 tests in 0.000s

OK


In [10]:
# @title Atom of Thought (AoT) Simulation: The AI Travel Agent
import time
import random

# ==============================================================================
# THE CORE IDEA:
# We will simulate an AI agent planning a trip. Instead of tackling the
# whole problem at once, it will break it down into atomic thoughts,
# process them independently, and then assemble the final plan.
# ==============================================================================

# THE COMPLEX USER REQUEST
# ------------------------
main_query = "Plan a 3-day weekend trip to Rome for a history buff on a budget of $800, leaving from New York."

# STAGE 1: DECOMPOSITION
# ----------------------
# An LLM would perform this step. Here, we simulate it by manually
# breaking the main query into independent, atomic questions.
def decompose_query(query):
    """Simulates an LLM breaking a complex query into atomic thoughts."""
    print("🧠 Decomposing the main query into atomic thoughts...")
    time.sleep(1)

    # These are the "Atoms of Thought"
    atomic_queries = [
        "Find budget round-trip flights from New York to Rome for a weekend.",
        "List 3 highly-rated, budget-friendly hotels or hostels in Rome near historical sites.",
        "Create a 3-day itinerary of historical sites in Rome, prioritizing the Roman Forum and Colosseum.",
        "Estimate the cost of food and local transport for 3 days in Rome.",
        "Check for any special historical exhibits or events in Rome for the upcoming weekend."
    ]

    print(f"   -> Found {len(atomic_queries)} atomic tasks to execute.\n")
    return atomic_queries

# STAGE 2: INDEPENDENT EXECUTION
# -------------------------------
# Each atomic query is now processed independently. In a real system,
# these could be run in parallel by different "AI workers".
def execute_atomic_query(atomic_query):
    """Simulates an AI worker processing a single, atomic query."""
    print(f"  ⚡ Executing atom: '{atomic_query}'")

    # Simulate the time it takes for research/API calls
    time.sleep(random.uniform(1.5, 3.0))

    # We use simple hardcoded responses to simulate the output for each atom.
    if "flights" in atomic_query.lower():
        return {
            "type": "Flights",
            "content": "Found budget flights with 'Vueling Air' for $450 (round-trip).",
            "cost": 450
        }
    elif "hotels" in atomic_query.lower():
        return {
            "type": "Accommodation",
            "content": "1. The Beehive Hostel (Rating: 4.5/5, ~$50/night)\n2. Hotel Romano (Rating: 4.2/5, ~$90/night)\n3. Generator Rome (Rating: 4.3/5, ~$55/night)",
            "cost": (55 * 3) # Assume Generator Hostel for 3 nights
        }
    elif "itinerary" in atomic_query.lower():
        return {
            "type": "Itinerary",
            "content": """
    - Day 1: Colosseum, Roman Forum & Palatine Hill.
    - Day 2: Vatican City (St. Peter's Basilica, Vatican Museums).
    - Day 3: Pantheon & Trevi Fountain.
            """
        }
    elif "food and transport" in atomic_query.lower():
        return {
            "type": "Daily Costs",
            "content": "Estimated daily cost for budget food (pizza/pasta) and public transport is ~$60/day.",
            "cost": (60 * 3)
        }
    elif "special exhibits" in atomic_query.lower():
        return {
            "type": "Special Events",
            "content": "No major special exhibits found, but the 'Night of Museums' event is happening this Saturday."
        }
    else:
        return {"type": "Unknown", "content": "Could not process this query."}

# STAGE 3: SYNTHESIS
# -------------------
# The results from all the independent atoms are now combined into a
# single, coherent, and useful response.
def synthesize_results(results):
    """Simulates the final step of combining all results into a coherent plan."""
    print("\n\n✨ Synthesizing all results into a final plan...")
    time.sleep(1)

    # Extracting data for summary
    flight_info = next(item for item in results if item['type'] == 'Flights')
    accommodation_info = next(item for item in results if item['type'] == 'Accommodation')
    daily_cost_info = next(item for item in results if item['type'] == 'Daily Costs')
    itinerary_info = next(item for item in results if item['type'] == 'Itinerary')
    events_info = next(item for item in results if item['type'] == 'Special Events')

    total_estimated_cost = flight_info['cost'] + accommodation_info['cost'] + daily_cost_info['cost']
    budget = 800

    # Building the final report using an f-string
    final_plan = f"""
==================================================================
              **Your Personalized Trip Plan to Rome**
==================================================================

Hello! Based on your request for a 3-day history-focused trip to Rome
on a budget of ${budget}, here is a plan created by breaking down your
request into smaller pieces and assembling the results.

---
**Budget Summary:**
---
  - ✈️ Flights:      ${flight_info['cost']}
  - 🏨 Accommodation: ${accommodation_info['cost']} (Est. for 3 nights at Generator Rome)
  - 🍝 Food & Transit: ${daily_cost_info['cost']} (Est. for 3 days)
  ---------------------------------
  - **Total Estimated Cost: ${total_estimated_cost}**
  - Your Budget:          ${budget}
  - **Remaining Budget:   ${budget - total_estimated_cost}**

---
**Recommended Itinerary for a History Buff:**
---
{itinerary_info['content']}

---
**Flight & Accommodation Details:**
---
*   **Flight:** {flight_info['content']}
*   **Accommodation Options:**
{accommodation_info['content']}

---
**Special Notes:**
---
*   {events_info['content']}

Enjoy your historical trip to Rome!
"""
    return final_plan

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    print("--- STARTING ATOM OF THOUGHT SIMULATION ---\n")

    # Stage 1
    atomic_tasks = decompose_query(main_query)

    # Stage 2
    print("--- STAGE 2: EXECUTING ATOMIC THOUGHTS ---")
    all_results = []
    for task in atomic_tasks:
        result = execute_atomic_query(task)
        all_results.append(result)
        print(f"    -> Result for '{result['type']}': Complete.\n")

    # Stage 3
    final_report = synthesize_results(all_results)

    print(final_report)
    print("--- SIMULATION COMPLETE ---")

--- STARTING ATOM OF THOUGHT SIMULATION ---

🧠 Decomposing the main query into atomic thoughts...
   -> Found 5 atomic tasks to execute.

--- STAGE 2: EXECUTING ATOMIC THOUGHTS ---
  ⚡ Executing atom: 'Find budget round-trip flights from New York to Rome for a weekend.'
    -> Result for 'Flights': Complete.

  ⚡ Executing atom: 'List 3 highly-rated, budget-friendly hotels or hostels in Rome near historical sites.'
    -> Result for 'Accommodation': Complete.

  ⚡ Executing atom: 'Create a 3-day itinerary of historical sites in Rome, prioritizing the Roman Forum and Colosseum.'
    -> Result for 'Itinerary': Complete.

  ⚡ Executing atom: 'Estimate the cost of food and local transport for 3 days in Rome.'
    -> Result for 'Unknown': Complete.

  ⚡ Executing atom: 'Check for any special historical exhibits or events in Rome for the upcoming weekend.'
    -> Result for 'Unknown': Complete.



✨ Synthesizing all results into a final plan...


StopIteration: 

In [11]:
# @title Live Atom of Thought (AoT) Simulation with Groq LLM
# ==============================================================================
# This cell simulates the AoT process using a live LLM at each stage.
# 1. DECOMPOSITION: The LLM breaks down a complex query.
# 2. EXECUTION: The LLM answers each of the smaller queries.
# 3. SYNTHESIS: The LLM assembles the answers into a final report.
# ==============================================================================

# 1. SETUP AND INSTALLATION
# =========================
!pip install -q groq
print("✅ 'groq' library installed.")

import json
from getpass import getpass
from groq import Groq

# 2. API KEY INPUT & CLIENT CONFIGURATION
# ========================================
client = None
try:
    api_key = getpass('🔑 Please enter your Groq API Key and press Enter: ')
    if not api_key:
        raise ValueError("API Key cannot be empty.")
    client = Groq(api_key=api_key)
    print("✅ Groq client configured successfully.")
except Exception as e:
    print(f"🛑 Configuration failed: {e}")

# 3. DEFINE THE CORE LLM INTERACTION FUNCTION
# ============================================
def call_groq_llm(system_prompt, user_prompt, is_json_output=False):
    """A helper function to call the Groq API and get a response."""
    if not client:
        return "Groq client not configured."

    try:
        response_kwargs = {
            "model": "llama3-70b-8192",
            "temperature": 0.2, # Lower temperature for more focused, less random output
        }
        if is_json_output:
            response_kwargs["response_format"] = {"type": "json_object"}

        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            **response_kwargs
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"An error occurred during API call: {e}"

# --- THE SIMULATION STARTS HERE ---

if client:
    # THE COMPLEX USER REQUEST
    main_query = "I want to plan a 4-day trip to Tokyo for a first-time visitor. My interests are a mix of modern tech (like Akihabara) and traditional culture (temples). My budget is $1500, traveling from San Francisco (SFO)."

    print("\n" + "="*50)
    print("      🚀 STARTING ATOM OF THOUGHT SIMULATION 🚀")
    print("="*50 + "\n")
    print(f"**Main Goal:** {main_query}\n")

    # ==============================================================================
    # STAGE 1: DECOMPOSITION (LLM breaks down the problem)
    # ==============================================================================
    print("--- 🧠 STAGE 1: DECOMPOSITION ---")
    print("Asking the LLM to break the main query into independent atomic tasks...")

    decomposition_system_prompt = "You are a master planning AI. Your job is to break down a complex user request into a list of simple, independent, and specific sub-tasks that can be researched separately. Provide the output as a JSON object with a single key: 'tasks'."
    decomposition_user_prompt = f"Please decompose this user request into atomic tasks: '{main_query}'"

    # Call the LLM and expect a JSON response
    decomposition_response = call_groq_llm(decomposition_system_prompt, decomposition_user_prompt, is_json_output=True)

    try:
        atomic_tasks = json.loads(decomposition_response).get("tasks", [])
        if not atomic_tasks:
            raise Exception("LLM did not return a list of tasks.")
        print(f"   -> ✅ LLM successfully decomposed the query into {len(atomic_tasks)} atomic tasks:\n")
        for i, task in enumerate(atomic_tasks, 1):
            print(f"      {i}. {task}")
    except Exception as e:
        print(f"   -> 🛑 Failed to get atomic tasks from LLM. Error: {e}")
        atomic_tasks = []


    # ==============================================================================
    # STAGE 2: EXECUTION (LLM processes each atom independently)
    # ==============================================================================
    if atomic_tasks:
        print("\n--- ⚡ STAGE 2: INDEPENDENT EXECUTION ---")
        print("Asking the LLM to execute each atomic task as a separate 'worker'...")

        execution_results = []
        execution_system_prompt = "You are an expert travel researcher. Provide a concise, direct, and factual answer to the following user question."

        for task in atomic_tasks:
            print(f"\n  > Executing atom: '{task}'")
            # Call the LLM for each individual task
            task_result = call_groq_llm(execution_system_prompt, task)
            execution_results.append({"task": task, "result": task_result})
            print(f"    -> LLM Response received.")
        print("\n   -> ✅ All atomic tasks have been executed.")


    # ==============================================================================
    # STAGE 3: SYNTHESIS (LLM assembles the final report)
    # ==============================================================================
    if 'execution_results' in locals() and execution_results:
        print("\n--- ✨ STAGE 3: SYNTHESIS ---")
        print("Giving all the research back to the LLM to assemble the final report...")

        synthesis_system_prompt = "You are a friendly and helpful AI travel agent. Your job is to synthesize a collection of research notes into a final, well-formatted, and coherent travel plan for the user. Do not make up information; only use the notes provided. Start with a friendly greeting."

        # Format the collected results into a single string for the prompt
        notes_for_synthesis = "\n".join([f"Task: {item['task']}\nResult: {item['result']}\n---" for item in execution_results])
        synthesis_user_prompt = f"Please create a final travel plan for the user based on the following research notes:\n\n**RESEARCH NOTES:**\n{notes_for_synthesis}"

        final_report = call_groq_llm(synthesis_system_prompt, synthesis_user_prompt)

        print("\n" + "="*50)
        print("             Final Synthesized Report")
        print("="*50)
        print(final_report)

    print("\n\n--- SIMULATION COMPLETE ---")

✅ 'groq' library installed.
🔑 Please enter your Groq API Key and press Enter: ··········
✅ Groq client configured successfully.

      🚀 STARTING ATOM OF THOUGHT SIMULATION 🚀

**Main Goal:** I want to plan a 4-day trip to Tokyo for a first-time visitor. My interests are a mix of modern tech (like Akihabara) and traditional culture (temples). My budget is $1500, traveling from San Francisco (SFO).

--- 🧠 STAGE 1: DECOMPOSITION ---
Asking the LLM to break the main query into independent atomic tasks...
   -> ✅ LLM successfully decomposed the query into 9 atomic tasks:

      1. {'id': 1, 'description': 'Determine the best time of year to visit Tokyo considering weather and tourist season', 'dependencies': []}
      2. {'id': 2, 'description': 'Research and book a round-trip flight from SFO to Tokyo within the budget of $1500', 'dependencies': [1]}
      3. {'id': 3, 'description': 'Find and book a hotel or accommodation in Tokyo that fits within the budget of $1500', 'dependencies': [2]}